# RI #5: Describe your data, and let the profile build the tools

`search`, `grep`, and `open` treat a corpus as a pile of text. Most corpora are not: posts carry
tags and scores, filings carry tables and numbered pages, a repository carries definitions. Here a
short YAML profile says which of those a corpus holds, and
`aieng.advanced_rag.agents.profile` turns the profile into tools on the same tool server, with the
same trace. The tool interface is the variable; the model, the corpus, and the questions stay fixed.

The notebook runs that idea on three public corpora: the ServerFault 10,000-question slice, the
84 FinanceBench filings as RI #3 ingested them, and the `click` library at a pinned commit. It
compares four arms on each:

| Arm | Tools | System prompt |
| --- | --- | --- |
| `G` | `C7`: `search` (k 20), `grep` inside what search returned, `open` | generic |
| `D` | the same as `G` | generic plus the profile's description of the data |
| `P` | `C7` plus every tool the profile generates | plus the description |
| `Q` | `open` plus the generated tools, with no `search` or `grep` | plus the description |

`D` separates describing the data from building tools for it. `Q` asks whether the generated
tools can replace the generic ones rather than add to them.

This is one run with no repeats: one seed, 24 questions per corpus, looking for a large effect of
the interface. The reference results ran on the on-prem `Qwen3_6-35B-A3B` and were judged by the
same model; with `RUN_EXPENSIVE = False` the notebook reads them from the cache, or from the
committed summary in `outputs/describe_your_data/results.json` when the cache is empty.

In [1]:
import json
import sys
from pathlib import Path

from aieng.advanced_rag import spend
from aieng.advanced_rag.agents.corpus_tools import CorpusToolServer
from aieng.advanced_rag.agents.profile import load_profile
from aieng.advanced_rag.agents.toolset import tool_declarations
from aieng.advanced_rag.models import choose_models


sys.path.insert(0, str(Path.cwd()))
import describe_your_data as dyd  # noqa: E402


RUN_EXPENSIVE = False
MODELS = choose_models(show=True)

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget not set on the key; assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override), spent $135.44, left $164.56


## A profile is a short description of a folder

A profile names the files that are documents or code, how documents split into sections,
identifier patterns, tables with described columns, and source code to index. Each part adds a
tool family:

| Profile part | Tools it adds |
| --- | --- |
| `identifiers` | `find_identifier` (documents holding one value, matched exactly), `list_identifiers` (distinct values of a kind with document counts) |
| `tables` | `sql`, a read-only SQLite query whose description carries the schema and column descriptions |
| `sections` on a file rule | `outline`, `open_section` |
| `code` (Python) | `find_symbol`, `open_symbol` |

The ServerFault profile, as committed:

In [2]:
print(dyd.profile_path("serverfault").read_text())

# Data profile for the ServerFault 10,000-question slice, as
# `describe_your_data.py prepare serverfault` writes it: one Markdown file per
# post and two tables of post metadata.
name: serverfault
description: >-
  ServerFault questions about running servers and networks, each with its accepted answer.
  Posts are written by administrators and quote commands, configuration lines, file paths,
  and error messages exactly as they appeared.

files:
  - glob: "posts/*.md"
    kind: document
    sections: markdown

identifiers:
  - name: unix_path
    pattern: '(?<![\w/.~-])/(?:etc|var|usr|opt|proc|sys|dev|home|srv|tmp|lib|boot|root|run|bin|sbin)(?:/[\w.+@-]+)+'
    description: a file-system path under a standard Unix folder
    examples: [/etc/nginx/nginx.conf, /var/log/messages]
  - name: constant
    pattern: '\b[A-Z][A-Z0-9]*_[A-Z0-9_]{2,}\b'
    description: an upper-case name with underscores, such as a socket state, registry hive, or variable
    examples: [TIME_WAIT, HKEY_LOCAL_MA

## Build the tools on the click repository

`prepare_click` downloads the pinned click archive (343 KB, checked against its SHA-256), extracts
it into the cache, and the profile turns it into a corpus and tools. Nothing calls a model, so this
cell runs with `RUN_EXPENSIVE = False`. The ServerFault and FinanceBench folders are built the same
way with `python describe_your_data.py prepare`, from the corpora RI #3 and RI #4 fetch.

In [3]:
if not dyd.corpus_folder("click").is_dir():
    dyd.prepare_click()
profile, corpus, tools = dyd.load("click")
print(json.dumps(tools.summary(), indent=1))

server = CorpusToolServer(corpus)
tools.register(server)
session = server.session(tools.arm(dyd.STANDARD_ARMS["C5"], name="P5"))
print([tool.__name__ for tool in session.tools])

{
 "documents": 65,
 "tools": [
  "outline",
  "open_section",
  "find_symbol",
  "open_symbol"
 ],
 "identifier_kinds": {},
 "tables": {},
 "documents_with_sections": 28,
 "sections": 219,
 "symbols": 999
}
['grep', 'open', 'outline', 'open_section', 'find_symbol', 'open_symbol']


Each generated tool is a plain function with a docstring, which is what the model sees as its
tool declaration. Two calls by hand:

In [4]:
found = session.call("find_symbol", {"name": "make_pass_decorator"})
print(json.dumps(found, indent=1)[:800])
opened = session.call("open_symbol", {"name": found["symbols"][0]["qualname"]})
print(opened["doc_id"], opened["lines"])
print(opened["text"][:600])
print(session.trace.counts(), session.trace.read_ids)

{
 "total": 1,
 "symbols": [
  {
   "qualname": "click.decorators.make_pass_decorator",
   "kind": "function",
   "doc_id": "src/click/decorators.py",
   "lines": "50-96",
   "signature": "def make_pass_decorator(",
   "summary": "Given an object type this creates a decorator that will work"
  }
 ]
}
src/click/decorators.py 50-96
50: def make_pass_decorator(
51:     object_type: t.Type[T], ensure: bool = False
52: ) -> t.Callable[["t.Callable[te.Concatenate[T, P], R]"], "t.Callable[P, R]"]:
53:     """Given an object type this creates a decorator that will work
54:     similar to :func:`pass_obj` but instead of passing the object of the
55:     current context, it will find the innermost context of type
56:     :func:`object_type`.
57: 
58:     This generates a decorator that works roughly like this::
59: 
60:         from functools import update_wrapper
61: 
62:         def decorator(f):
63:             @pass_context

{'find_symbol': 1, 'open_symbol': 1} ['src/click/decorators.py']


In [5]:
print(json.dumps(tool_declarations([next(t for t in session.tools if t.__name__ == "open_section")])[0], indent=1))

{
 "description": "Read one section of a document in full, with line numbers.\n\nParameters\n----------\ndoc_id : str\n    A ``doc_id`` from an earlier result.\nsection : str\n    The section's number from outline, or its title (an exact title\n    wins, then the first title containing the text, ignoring case).\n\nReturns\n-------\ndict\n    The section's ``title``, ``lines``, and ``text``. A long section stops\n    early with a ``note`` naming the line to continue from.",
 "name": "open_section",
 "parameters_json_schema": {
  "properties": {
   "doc_id": {
    "title": "Doc Id",
    "type": "string"
   },
   "section": {
    "title": "Section",
    "type": "string"
   }
  },
  "required": [
   "doc_id",
   "section"
  ],
  "title": "open_sectionParams",
  "type": "object"
 }
}


## The designs considered, and the four run

Five designs that differ in kind were on the table before any run:

1. Add the generated tools to the generic arm (`P` against `G`): does the interface add anything?
2. Replace the generic tools with the generated ones (`Q`): can a data-specific interface stand alone?
3. Describe the data in the prompt and change no tool (`D`): is the gain in the tools or in the description?
4. Change the unit of the corpus instead of the tools, such as one document per filing section
   rather than per filing.
5. A deliberately misleading profile (wrong column descriptions), as a negative control that must lose.

Designs 1 to 3 cost the same question set and isolate one change each, so they ran. Designs 4 and 5
are recorded below as the next experiments.

The question sets: ServerFault has 12 questions computed from the post metadata and the path index
(how many posts carry a tag or two tags, the top-scored post in a tag, the post that mentions a
path) and 12 from the synthetic Q&A path over sampled posts. FinanceBench reuses 24 of its own
questions, 8 of each type. click has 16 questions read from the source with `ast` (parameter
defaults, base classes, defining files) and 8 from the synthetic Q&A path over the docs. Every
answer is judged against its reference with the shared judge prompt in
`aieng.advanced_rag.evaluation.answers`.

## The arm table

With `RUN_EXPENSIVE = True` and no result of yours, the arms run on `MODELS.generation` and are
judged by `MODELS.judging`. The family's cheap tier runs all 24 questions per corpus; a mid or
frontier model is capped at 10 per corpus, spread over the question types. A cheap-tier live run
is four arms times 24 questions times three corpora, or 288 agent runs; the reference used about
2,800 model calls, half of them on FinanceBench, and the cell below prints the estimate of
rebuilding it on your model. Read it before you start a run in hands-on notebook time.

In [6]:
rows, origin = dyd.results(MODELS, live=RUN_EXPENSIVE)
print(origin)
header = f"{'corpus':13} {'arm':3} {'model':16} {'n':>3} {'acc':>6} {'calls':>6} {'in tok':>8} {'out tok':>7} {'exh':>3} {'gold read':>9}"
print(header)
for r in rows:
    print(
        f"{r['corpus']:13} {r['arm']:3} {r['model'][:16]:16} {r['questions']:3d} {r['accuracy']:6.3f} "
        f"{r['calls']:6.2f} {r['prompt_tokens']:8,.0f} {r['completion_tokens']:7,.0f} {r['exhausted']:3d} "
        f"{r['gold_read']:9.2f}"
    )

describe-your-data-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                        861 calls, 6,382,432 prompt + 67,023 completion tokens
                        rebuilding with gemini-3.5-flash-lite: about 861 calls, est. $2.08 (set RUN_EXPENSIVE = True to rebuild)
describe-your-data-judgements reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                              87 calls, 25,279 prompt + 4,109 completion tokens
                              rebuilding with gemini-3.8-flash: about 87 calls, est. $0.03 (set RUN_EXPENSIVE = True to rebuild)


committed reference summary (Qwen3_6-35B-A3B, judged by Qwen3_6-35B-A3B); the local cache had none: no describe-your-data-runs for gemini-3.5-flash-lite or for the reference model Qwen3_6-35B-A3B in the cache; fetch the reference artifacts or set RUN_EXPENSIVE = True to build it
corpus        arm model              n    acc  calls   in tok out tok exh gold read
serverfault   G   Qwen3_6-35B-A3B   24  0.542  10.75  111,252     834   4      0.17
serverfault   D   Qwen3_6-35B-A3B   24  0.500   9.67   86,323   1,072   1      0.22
serverfault   P   Qwen3_6-35B-A3B   24  0.917   5.54   27,819     275   0      0.33
serverfault   Q   Qwen3_6-35B-A3B   24  0.875   9.92   40,541     612   2      0.33
financebench  G   Qwen3_6-35B-A3B   24  0.708  14.62  278,966   1,402   3      0.79
financebench  D   Qwen3_6-35B-A3B   24  0.542  15.25  304,363   1,264   6      0.83
financebench  P   Qwen3_6-35B-A3B   24  0.833  13.38  170,254   1,303   3      0.96
financebench  Q   Qwen3_6-35B-A3B   24  0.750  1

`acc` is the share the judge marked correct, `calls` and the token columns are per question,
`exh` counts runs that reached the 30-call budget, and `gold read` is the share of questions with
a known source document whose document the agent opened (counting questions excluded).

In [7]:
for r in rows:
    print(f"{r['corpus']:13} {r['arm']}  " + "  ".join(f"{t} {v:.2f}" for t, v in r["by_type"].items()))

serverfault   G  count 0.00  count_two_tags 0.00  identifier 0.33  single_hop 0.92  top_score 0.33
serverfault   D  count 0.00  count_two_tags 0.00  identifier 0.33  single_hop 0.92  top_score 0.00
serverfault   P  count 1.00  count_two_tags 1.00  identifier 1.00  single_hop 0.83  top_score 1.00
serverfault   Q  count 1.00  count_two_tags 1.00  identifier 1.00  single_hop 0.75  top_score 1.00
financebench  G  domain-relevant 0.88  metrics-generated 0.50  novel-generated 0.75
financebench  D  domain-relevant 0.75  metrics-generated 0.25  novel-generated 0.62
financebench  P  domain-relevant 0.88  metrics-generated 0.75  novel-generated 0.88
financebench  Q  domain-relevant 0.75  metrics-generated 0.62  novel-generated 0.88
click         G  base_class 1.00  default_value 1.00  defining_file 1.00  single_hop 1.00
click         D  base_class 0.80  default_value 1.00  defining_file 1.00  single_hop 1.00
click         P  base_class 1.00  default_value 1.00  defining_file 1.00  single_hop 1.0

In [8]:
for r in rows:
    if r["arm"] in ("P", "Q"):
        print(f"{r['corpus']:13} {r['arm']}  {r['tool_mix']}")

serverfault   P  {'open': 69, 'search': 25, 'sql': 18, 'grep': 17, 'find_identifier': 2}
serverfault   Q  {'sql': 139, 'open': 75, 'find_identifier': 5, 'list_identifiers': 2}
financebench  P  {'sql': 111, 'grep': 62, 'open': 48, 'open_section': 28, 'outline': 28, 'search': 19}
financebench  Q  {'sql': 214, 'open_section': 54, 'outline': 46, 'open': 24}
click         P  {'find_symbol': 17, 'open_symbol': 12, 'open': 11, 'search': 10, 'open_section': 5, 'outline': 3, 'grep': 2}
click         Q  {'find_symbol': 31, 'outline': 21, 'open_symbol': 18, 'open_section': 9, 'open': 7}


## What the reference run found

One run, one seed, 24 questions per corpus, `Qwen3_6-35B-A3B` answering and judging, so read
differences of one or two questions as noise.

The surprising result first: describing the data in the prompt did nothing on its own. Arm `D`
scored one question lower than `G` on ServerFault (12 against 13), four lower on FinanceBench (13
against 17), and one lower on click (23 against 24), and on FinanceBench it read more tokens.
Every gain below came with the generated tools, not with the description.

- ServerFault: `P` answered 22 of 24 against 13 for `G`. All of the gain is on the 12 questions
  computed from post metadata and the path index: `G` answered 2 of 12 and `P` all 12. The
  generic tools could have counted tags, because each post has a `Tags:` line and `grep` reports
  how many documents match, but the model never tried that. On the 12 synthetic questions written
  from post text, every arm answered 9 to 11. `P` also used 5.5 calls and 28,000 prompt tokens per
  question against 10.8 calls and 111,000 tokens for `G`.
- FinanceBench: `P` answered 20 of 24 against 17 for `G`, with the gain on the
  `metrics-generated` questions (6 of 8 against 4 of 8), which ask for one line item from a
  financial statement. `P` read 170,000 prompt tokens per question against 279,000, and both arms
  reached the 30-call budget on 3 questions.
- click: accuracy is at the ceiling (`G` and `P` both 24 of 24), so the interface shows up only in
  cost: `P` took 3.7 calls and 13,500 prompt tokens per question against 8.1 calls and 44,000 for
  `G`, about a third of the tokens.
- `Q`, the generated tools without `search` and `grep`, beat `G` on ServerFault and FinanceBench
  but lost to `P` everywhere, and lost questions on prose: 9 of 12 synthetic ServerFault questions
  against 10 for `P`, and 6 of 8 click documentation questions against 8.

These results are scoped to this model, these three profiles, and these question sets. The
ServerFault metadata questions were built from the same tables the profile exposes, so they are an
existence proof that the tools can matter, not an estimate of how often your questions need them.

Where the profile tools did not win (the prose questions on ServerFault and click), two things
would have to be true for them to win: the questions would need an exact handle the generic tools
blur, such as an identifier or a table value, or the corpus would have to be large enough that
`search` at k 20 misses the right document. The cheapest tests are a misleading profile (design 5,
which must lose if the description is read at all) and the same click questions on a larger
repository, where `grep` returns too many hits for the generic arm to read.

## Describe your own folder

Copy `profiles/template.yaml` next to your documents, edit it, and point the tool server at the
folder. The template explains every key. Then run the arms on five of your own questions, with
`RUN_EXPENSIVE = True`, before trusting any of them.

In [9]:
print((dyd.PROFILES / "template.yaml").relative_to(dyd.HERE))
template = load_profile(dyd.PROFILES / "template.yaml")
print(template.structures)

profiles/template.yaml
('identifiers', 'tables', 'sections', 'code')


In [10]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
this run, proxy usage $0.00 (team key: includes teammates' calls in the same window)
key budget not set on the key; assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override), spent $135.44, left $164.56
